# LOW thinking: bill splitting

Rerun of the original dataset and evaluator with explicit LOW thinking. Usage records distinguish prompt, final-output, thinking and total tokens. Missing fields stay null. Tool creation is measured separately from task execution. The optional comparison later uses fresh MEDIUM calls and an explicitly requested Gemini-written tool; that arm is an ablation, not autonomous self-improvement.


## Recorded results — October 6, 2026

| Held-out arm | Passed / attempts | Tool calls | Input | Output | Reported thinking* | Total | Usage coverage |
|---|---:|---:|---:|---:|---:|---:|---:|
| LOW_no_tool | 48/48 | 0 | 5,464 | 4,538 | 7,763 | 17,765 | 48/48 |
| MEDIUM_no_tool | 38/48 | 0 | 4,347 | 4,211 | 19,236 | 27,794 | 38/48 |
| LOW_tool_available | 48/48 | 40 | 7,528 | 4,814 | 7,355 | 19,697 | 48/48 |
| MEDIUM_tool_available | 48/48 | 22 | 7,528 | 4,658 | 55,501 | 67,687 | 48/48 |
| LOW_autonomous_tool_policy | 48/48 | 48 | 8,248 | 5,257 | 2,888 | 16,393 | 48/48 |

*Missing thinking-count fields remain null in the raw record. The table sums only reported counts. For all completed responses here, total minus input minus output reconciles with reported thinking (or zero when the field is omitted). LOW is not a fixed token budget. MEDIUM request failures have no returned usage; their totals are incomplete and cannot be compared as equal-work cost totals.

The first baseline passed 48/48 development attempts. Both improver proposals chose no change; their further repetitions exposed two genuine arithmetic failures, giving **142/144** on the same policy. A fixed improver given those development failures voluntarily wrote a Decimal calculator and instructed the task agent to call it. It passed **48/48 development and 48/48 held-out** attempts. The held-out original also passed 48/48, so this does not demonstrate a held-out accuracy gain.

The autonomous tool policy used **16,393 vs 17,765** held-out tokens: **7.7% fewer**, with 2,888 vs 7,763 reported thinking tokens. Creation cost **40,335 tokens**; at the observed saving, about **1,412 future bills** repay creation alone, or **1,964** including its 15,799-token development evaluation. These extrapolations assume the small-sample average savings continue and ignore monetary pricing, cache effects and CPU costs.

The controlled tool-available arm preserved the original task instruction and used **19,697 tokens**, more than no-tool LOW. It called the tool on 40/48 tasks. Thus writing a correct tool and teaching a policy to use it efficiently are separate changes. MEDIUM no-tool had four HTTP 503 failures and six read timeouts; all 38 returned answers were correct. This limits conclusions about thinking level and accuracy.

The complete notebook attempted **580 calls** and returned **357,871 reported tokens**, including proposals, development and all comparison arms. Repeated attempts are not independent new tasks. The fixed improver was never modified; this is not recursive improvement.

Configuration: `gemini-3.8-flash`, `google-genai==2.28.0`, explicit LOW/MEDIUM, temperature 0, output cap 8,192, request timeout 90 seconds, no retries. The output cap includes thinking. These data test LOW versus MEDIUM; they do not test HIGH. Sources: [Google thinking configuration](https://ai.google.dev/gemini-api/docs/thinking), [SDK types and usage fields](https://googleapis.github.io/python-genai/genai.html#genai.types.GenerateContentResponseUsageMetadata).

The bill response schema still asks for a short calculation summary and an amount even on tool calls. Some responses therefore calculate alongside calling the tool. These are observed end-to-end policy costs, not a clean measurement of replacing all internal arithmetic with argument routing. Tool execution itself consumes CPU time, not Gemini tokens, in this harness.


In [ ]:
%pip -q install google-genai==2.28.0

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

## 1. Bills, evaluator, and a small tool boundary
We generate 48 bills: ordinary amounts, fractional rates, and rounding edges.
Each group contributes eight development and eight held-out bills. Each bill gets
two independent attempts at temperature zero; repeats reveal run variability,
not independent new tasks. The half-up rule and tip basis are explicit.

The objective is exact per-person cents, with one model call per bill. Tokens,
request latency, and tool usage are reported separately. A tool request returns
its executed result directly as the final answer; there is no second model call.
This simple JSON tool protocol is part of the fixed harness.

The following helper accepts only a small arithmetic Python grammar. It is a toy
validator, not a general code sandbox. The oracle is never exposed to the task
agent, and the improver never sees held-out bills or answers.

In [4]:
import os, ast, re, json, random, time, statistics
from decimal import Decimal, ROUND_HALF_UP
from concurrent.futures import ThreadPoolExecutor
from google import genai
from google.genai import types

MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
REPEATS, WORKERS = 2, 6
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"], vertexai=False,
    http_options=types.HttpOptions(timeout=90000,
        retry_options=types.HttpRetryOptions(attempts=1)))
THINKING_LEVEL = "LOW"
usage, calls = [], []

def ask(prompt, schema):
    start = time.monotonic()
    call = {"attempt":len(calls)+1, "thinking_level":THINKING_LEVEL}
    calls.append(call)
    try:
        response = client.models.generate_content(model=MODEL, contents=prompt,
            config={"temperature":0, "max_output_tokens":8192,
                    "thinking_config":{"thinking_level":THINKING_LEVEL},
                    "response_mime_type":"application/json", "response_json_schema":schema,
                    "automatic_function_calling":{"disable":True}})
        u = response.usage_metadata
        resources = {"tokens":u.total_token_count, "prompt_tokens":u.prompt_token_count,
            "output_tokens":u.candidates_token_count, "thinking_tokens":u.thoughts_token_count,
            "cached_tokens":u.cached_content_token_count,
            "seconds":round(time.monotonic()-start, 3)}
        # Preserve missing metadata as null, not zero.
        usage.append(resources)
        reason = str(response.candidates[0].finish_reason) if response.candidates else "no candidate"
        call.update(resources, finish_reason=reason)
        if not response.candidates or response.candidates[0].finish_reason != "STOP":
            error = RuntimeError("Incomplete model response: " + reason)
            error.resources = resources
            raise error
        return json.loads(response.text), resources
    except Exception as error:
        call.update(error_type=type(error).__name__, status=getattr(error,"code",None),
                    seconds=round(time.monotonic()-start,3))
        raise

def expected(task):
    s, tax, tip = (Decimal(task[k]) for k in ("subtotal", "tax", "tip"))
    tax_amount = s * tax / 100
    tip_basis = s if task["tip_on"] == "subtotal" else s + tax_amount
    return ((s + tax_amount + tip_basis * tip / 100) / task["diners"]).quantize(
        Decimal("0.01"), rounding=ROUND_HALF_UP)

def make_bills():
    rng = random.Random(20261005)
    development, held_out = [], []
    for group in ("ordinary", "fractional", "rounding"):
        for i in range(16):
            s = str(Decimal(rng.randrange(501, 500000)) / 100)
            tax, tip = rng.choice(["0", "5", "8", "10"]), rng.choice(["10", "15", "20"])
            diners = rng.randrange(2, 17)
            if group == "fractional":
                tax, tip = rng.choice(["7.25", "8.875", "9.125"]), rng.choice(["17.5", "18.25", "19.625"])
            if group == "rounding":
                # Includes exact half-cent ties, plus non-terminating divisions.
                s = str(Decimal(2 * rng.randrange(500, 10000) + 1) / 100)
                tax, tip, diners = "0", "0", (2 if i % 2 == 0 else 3)
            task = {"id": f"{group}-{i:02}", "group": group, "subtotal": s,
                    "tax": tax, "tip": tip, "diners": diners,
                    "tip_on": "subtotal" if i % 2 == 0 else "total"}
            (development if i < 8 else held_out).append(task)
    return development, held_out

development, held_out = make_bills()
assert not ({t["id"] for t in development} & {t["id"] for t in held_out})
assert expected({"subtotal":"10.01", "tax":"0", "tip":"0", "diners":2,
                 "tip_on":"subtotal"}) == Decimal("5.01")
print(MODEL, "|", len(development), "development bills |", len(held_out), "held-out bills")

def run_tool(source, name, arguments, extra=None):
    tree = ast.parse(source)
    allowed = (ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Assign,
        ast.Return, ast.Expr, ast.Name, ast.Load, ast.Store, ast.Constant,
        ast.BinOp, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.IfExp, ast.Compare,
        ast.Eq, ast.Call, ast.Attribute, ast.keyword)
    nodes = list(ast.walk(tree))
    if len(source) > 2500 or len(nodes) > 220 or len(tree.body) != 1:
        raise ValueError("Tool exceeds the small arithmetic grammar")
    fn = tree.body[0]
    if not isinstance(fn, ast.FunctionDef) or fn.name != name or fn.decorator_list:
        raise ValueError("Expected exactly the requested function")
    if fn.args.defaults or fn.args.kw_defaults or fn.args.vararg or fn.args.kwarg:
        raise ValueError("No defaults or variadic arguments")
    approved = {"Decimal", "int", "str"} | set(extra or {})
    for node in nodes:
        if not isinstance(node, allowed):
            raise ValueError("Disallowed Python syntax: " + type(node).__name__)
        if isinstance(node, (ast.Name, ast.arg)) and "__" in getattr(node, "id", getattr(node, "arg", "")):
            raise ValueError("Disallowed name")
        if isinstance(node, ast.Attribute) and node.attr != "quantize":
            raise ValueError("Only Decimal.quantize is allowed")
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id not in approved:
                raise ValueError("Disallowed function call")
            if not isinstance(node.func, (ast.Name, ast.Attribute)):
                raise ValueError("Disallowed call target")
        if isinstance(node, ast.Constant) and len(str(node.value)) > 100:
            raise ValueError("Constant too large")
    scope = {"__builtins__": {}, "Decimal": Decimal, "int": int, "str": str,
             "ROUND_HALF_UP": ROUND_HALF_UP, **(extra or {})}
    exec(compile(tree, "generated_arithmetic_tool", "exec"), scope)
    return scope[name](*arguments), scope[name]

gemini-3.8-flash | 24 development bills | 24 held-out bills


## 2. Collect baseline trajectories
Every record includes the request, response, displayed calculation summary,
reference answer, correctness, and resource usage. Errors count as unsuccessful
attempts; API failures are flagged separately and never described as arithmetic
failures. Responses are not hidden internal reasoning.

In [6]:
PARAMS = {"subtotal": {"type":"string"}, "tax": {"type":"string", "description":"Bare percent number, without a percent sign"},
          "tip": {"type":"string", "description":"Bare percent number, without a percent sign"}, "diners": {"type":"integer"},
          "tip_on": {"type":"string", "enum":["subtotal", "total"]}}
TASK_SCHEMA = {"type":"object", "properties":{
    "action":{"type":"string", "enum":["answer", "tool"]},
    "amount":{"type":"string", "description":"Dollar amount as a decimal numeric string"}, "calculation":{"type":"string"},
    "arguments":{"type":"object", "properties":PARAMS,
                 "required":list(PARAMS), "additionalProperties":False}},
    "required":["action", "amount", "calculation"], "additionalProperties":False}
BASELINE = {"instruction":"Calculate the bill accurately and answer the user's request.",
            "tool_code":"", "change":"no_change"}

def request(task):
    basis = "subtotal before tax" if task["tip_on"] == "subtotal" else "total after tax"
    return (f"Bill subtotal USD {task['subtotal']}; tax {task['tax']}%; "
        f"tip {task['tip']}% on the {basis}; split equally among {task['diners']} diners. "
        "Keep full precision in intermediate amounts; round only the final per-person "
        "amount to cents using ROUND_HALF_UP (ties round up).")

def trajectory(job):
    artifact, task, repeat = job
    record = {"id":task["id"], "group":task["group"], "repeat":repeat,
              "request":request(task), "reference":str(expected(task))}
    tool = artifact.get("tool")
    contract = ("You may use split_bill(subtotal, tax, tip, diners, tip_on), with dollar "
        "and percentage strings and an integer diner count. Return action='tool' "
        "with arguments, or action='answer' with amount. The host uses the tool result "
        "directly as the final answer." if tool else
        "No tools exist. Return action='answer' and compute the dollar amount yourself.")
    try:
        answer, resources = ask(artifact["instruction"] + "\n" + contract +
            "\nGive a short calculation summary (at most 40 words), not private reasoning.\n" +
            record["request"], TASK_SCHEMA)
        record.update(response=answer, **resources)
    except Exception as error:
        record.update(correct=False, api_error=type(error).__name__, error_detail=str(error) if isinstance(error, RuntimeError) else None, **getattr(error,"resources",{}))
        return record
    try:
        if answer["action"] == "tool":
            if tool is None:
                raise ValueError("No tool is available")
            args = answer["arguments"]
            amount = tool(*(args[k] for k in PARAMS))
            record["tool_result"] = str(amount)
        else:
            text = answer["amount"].strip()
            # Accept ordinary currency rendering without hiding arithmetic errors.
            if not re.fullmatch(r"(?:USD\s*|\$)?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?", text):
                raise ValueError("Unparseable amount")
            amount = Decimal(text.removeprefix("USD").strip().removeprefix("$").replace(",", ""))
        record.update(amount=str(amount), correct=(amount == expected(task)))
    except Exception as error:
        record.update(correct=False, execution_error=type(error).__name__)
    return record

def evaluate(artifact, tasks):
    jobs = [(artifact,t,r) for t in tasks for r in range(REPEATS)]
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        return list(pool.map(trajectory, jobs))

def summary(rows):
    return {"correct":sum(r["correct"] for r in rows), "attempts":len(rows),
        "bills":len({r["id"] for r in rows}),
        "tool_calls":sum(r.get("response",{}).get("action")=="tool" for r in rows),
        "api_errors":sum("api_error" in r for r in rows),
        "tokens":sum((r.get("tokens") or 0) for r in rows),
        "median_seconds":round(statistics.median(r.get("seconds",0) for r in rows),2)}

baseline_dev = evaluate(BASELINE, development)
print("Development baseline:", summary(baseline_dev))
for row in baseline_dev:
    if not row["correct"]:
        print(json.dumps(row))

Development baseline: {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 18127, 'median_seconds': 3.75}


## 3. Let the fixed improver choose
We use the same evidence for two proposals: an instruction-only change and an
open choice that can also write a tool or leave the agent unchanged. We give no
repair labels and never insist that a tool is better. The same fixed improver
instructions generate both proposals.

Generated tools must pass independent contract checks before any evaluation.
Candidates are retained only if they improve development accuracy, with no API
failures in the comparison. A tie retains the original. This conservative rule
makes “no demonstrated improvement” possible even when a proposal is sensible.

In [8]:
PROPOSAL_SCHEMA = {"type":"object", "properties":{
    "change":{"type":"string", "enum":["no_change", "instruction", "tool"]},
    "instruction":{"type":"string"}, "tool_code":{"type":"string"},
    "reason":{"type":"string"}}, "required":["change","instruction","tool_code","reason"],
    "additionalProperties":False}
IMPROVER = """You improve a bill-solving TASK agent. You cannot change yourself,
its model, evaluator, objective, or harness. Diagnose the supplied real development
trajectories, including correct answers. Do not invent failures. Choose no_change,
an instruction revision, or a reusable arithmetic tool. A tool is optional and
must be justified by evidence. The objective is exact cents under one model call
per bill. Keep task instructions under 150 words. For no_change, preserve the
original instruction and return empty tool_code. For instruction, return empty
tool_code. For tool, write exactly one Python function
split_bill(subtotal, tax, tip, diners, tip_on), taking dollar and percent strings,
an integer diners, and tip_on 'subtotal' or 'total'. Return the Decimal per-person
amount, with full intermediate precision and final ROUND_HALF_UP cents.
Provided: Decimal, str, int, ROUND_HALF_UP. Allowed: assignments, arithmetic,
conditional expressions, comparisons, Decimal.quantize. No imports, loops,
defaults, annotations, decorators, markdown fences, or other functions. Code must
be under 2500 characters. Explain the evidence for the choice in under 100 words.
"""

def propose(scope):
    proposal, resources = ask(IMPROVER + f"\nAllowed changes for this proposal: {scope}." +
        "\nOriginal agent: " + json.dumps(BASELINE) +
        "\nDevelopment trajectories: " + json.dumps(baseline_dev), PROPOSAL_SCHEMA)
    print(proposal["change"], "|", proposal["reason"])
    proposal["proposal_usage"] = resources
    if scope == "no_change or instruction" and proposal["change"] == "tool":
        proposal["validation_error"] = "Tool forbidden in instruction-only arm"
    elif proposal["change"] == "tool":
        try:
            for task in development:
                args = tuple(task[k] for k in PARAMS)
                result, tool = run_tool(proposal["tool_code"], "split_bill", args)
                assert result == expected(task), "Tool failed an independent contract check"
            proposal["tool"] = tool
        except Exception as error:
            proposal["validation_error"] = type(error).__name__ + ": " + str(error)
    elif proposal["tool_code"]:
        proposal["validation_error"] = "Unexpected code in a non-tool proposal"
    if proposal["change"] == "no_change":
        proposal["instruction"] = BASELINE["instruction"]
    return proposal

instruction_proposal = propose("no_change or instruction")
open_proposal = propose("no_change, instruction, or tool")
proposals = {"instruction_only":instruction_proposal, "open_choice":open_proposal}
artifacts, dev_rows = {"original":BASELINE}, {"original":baseline_dev}
for name, proposal in proposals.items():
    artifacts[name] = BASELINE if "validation_error" in proposal else proposal
    dev_rows[name] = evaluate(artifacts[name], development)
    print(name, summary(dev_rows[name]), "| validation:", proposal.get("validation_error","passed"))

# Decide retention before opening the held-out evaluation.
retained_name = "original"
if not any(summary(rows)["api_errors"] for rows in dev_rows.values()):
    for name in proposals:
        if sum(r["correct"] for r in dev_rows[name]) > sum(r["correct"] for r in dev_rows[retained_name]):
            retained_name = name
print("Retained:", retained_name)

no_change | All 48 runs across ordinary, fractional, and rounding groups achieved 100% accuracy (correct: true for every item). There are no failures to fix.
no_change | All 48 runs across ordinary, fractional, and rounding groups achieved 100% accuracy with exact cents under one model call per bill. Without observed failures, no change is warranted.
instruction_only {'correct': 47, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 17994, 'median_seconds': 3.62} | validation: passed
open_choice {'correct': 47, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 17768, 'median_seconds': 3.79} | validation: passed
Retained: original


## 4. Evaluate on unseen bills
We now evaluate all three frozen versions on the same held-out bills, including
rejected candidates as diagnostic comparisons. Held-out results never choose or
revise a version. The retained version was decided above. Error counts, paired
wins/losses, and results by bill group are more informative than one example.

In [10]:
held_out_rows = {name:evaluate(artifact, held_out) for name,artifact in artifacts.items()}
print("HELD-OUT RESULTS")
for name, rows in held_out_rows.items():
    print(name, summary(rows))
    for group in ("ordinary", "fractional", "rounding"):
        subset = [r for r in rows if r["group"] == group]
        print(" ", group, f"{sum(r['correct'] for r in subset)}/{len(subset)}")
    if name != "original":
        old = held_out_rows["original"]
        wins = sum(not a["correct"] and b["correct"] for a,b in zip(old,rows))
        losses = sum(a["correct"] and not b["correct"] for a,b in zip(old,rows))
        print("  paired wins/losses against original:", wins, losses)
print("Retained version:", retained_name)
print("Total calls:", len(usage), "| Reported tokens:", sum((r["tokens"] or 0) for r in usage))
# Everything needed to inspect or export trajectories remains in this session.
experiment = {"model":MODEL, "repeats":REPEATS, "retained":retained_name,
    "proposals":{k:{a:b for a,b in v.items() if a != "tool"} for k,v in proposals.items()},
    "development":dev_rows, "held_out":held_out_rows, "usage":usage}

HELD-OUT RESULTS
original {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 17765, 'median_seconds': 3.88}
  ordinary 16/16
  fractional 16/16
  rounding 16/16
instruction_only {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 18236, 'median_seconds': 3.58}
  ordinary 16/16
  fractional 16/16
  rounding 16/16
  paired wins/losses against original: 0 0
open_choice {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 18017, 'median_seconds': 3.53}
  ordinary 16/16
  fractional 16/16
  rounding 16/16
  paired wins/losses against original: 0 0
Retained version: original
Total calls: 290 | Reported tokens: 135430


## Read the result carefully
An accuracy gain on unseen bills supports this particular agent change. A tie
means we have not demonstrated an accuracy improvement. Tools may change token
cost or latency even on ties; proposal and development costs also belong in the
accounting. There are only 24 distinct held-out tasks, despite 48 attempts.

This synthetic experiment uses one model, one seed, one fixed improver, a known
formula, and a small action space. It does not show broad transfer or recursive
self-improvement. A stronger follow-up would repeat the whole experiment with
fresh seeds and independent proposals. First inspect the actual trajectories.

If both proposals choose `no_change`, the comparison arms are repeat evaluations
of the same policy. Their differences in token usage and latency are run
variability, not measured benefits of an agent change.


## Optional: inspect the recorded trajectories
Expand this JSON record to see actual responses and proposals. It contains no
credentials. Re-running this cell displays the current session's experiment.


In [13]:
from IPython.display import JSON, display
display(JSON(experiment, expanded=False))


In [14]:
experiment.update(thinking_level=THINKING_LEVEL, calls=calls)


## 5. Thinking versus executing a Gemini-written tool

This is a controlled 2×2 comparison: LOW/MEDIUM thinking and tool unavailable/available. The tool is explicitly requested from Gemini using development evidence. Its creation cost is separate; offering a tool does not force its use. Task instructions remain the original baseline, isolating the tool intervention. A simple JSON action protocol executes a call locally and returns its result directly; no second LLM synthesis call is charged. Real multi-turn agents may have additional overhead.

The efficiency candidate is selected on development data: no request failures, no accuracy regression, lower total tokens. Held-out tasks test it, never select it. This small sample cannot establish non-inferiority. For missing thought counts, `unattributed_tokens` reports total minus prompt minus output, rather than pretending the API reported zero.


In [15]:
# Controlled comparison: an explicitly requested Gemini-written tool.
# This is separate from the autonomous open-choice proposal above.
TOOL_WRITER = IMPROVER + """
For this controlled ablation, write the reusable split_bill tool even if all
answers were correct. The question is whether execution can replace thinking
cost while preserving accuracy. Do not claim observed arithmetic failures unless
there are actual failures. Return change='tool' and preserve the original task
instruction. Tool creation cost is paid once and reported separately.
"""
tool_proposal, creation_usage = ask(TOOL_WRITER + "\nDevelopment trajectories: " +
    json.dumps(baseline_dev), PROPOSAL_SCHEMA)
tool_artifact = dict(BASELINE, tool_code=tool_proposal["tool_code"])
for task in development:
    result, fn = run_tool(tool_artifact["tool_code"], "split_bill", tuple(task[k] for k in PARAMS))
    assert result == expected(task), "Generated tool failed development checks"
tool_artifact["tool"] = fn
print("Generated tool:",tool_proposal["tool_code"])
print("Creation usage:",creation_usage)

def costs(rows):
    result = summary(rows)
    for field in ("prompt_tokens","output_tokens","thinking_tokens","tokens"):
        observed=[r[field] for r in rows if r.get(field) is not None]
        result[field] = sum(observed) if observed else None
        result[field+"_reported_attempts"] = len(observed)
    result["tool_calls"] = sum(r.get("response",{}).get("action")=="tool" for r in rows)
    result['unattributed_tokens'] = sum(r['tokens']-r['prompt_tokens']-r['output_tokens'] for r in rows if all(r.get(k) is not None for k in ('tokens','prompt_tokens','output_tokens')))
    return result

# Development data select the efficiency candidate before held-out results.
low_tool_dev = evaluate(tool_artifact, development)
b, t = costs(baseline_dev), costs(low_tool_dev)
efficiency_retained = (not b['api_errors'] and not t['api_errors'] and
    t['correct'] >= b['correct'] and t['tokens'] is not None and b['tokens'] is not None
    and t['tokens'] < b['tokens'])
print("LOW tool development:",t,"| efficiency candidate retained:",efficiency_retained)
comparison = {"LOW_no_tool":held_out_rows["original"]}
for level, label, artifact in [("MEDIUM","MEDIUM_no_tool",BASELINE),
                              ("LOW","LOW_tool_available",tool_artifact),
                              ("MEDIUM","MEDIUM_tool_available",tool_artifact)]:
    THINKING_LEVEL = level
    comparison[label] = evaluate(artifact, held_out)
    print(label,costs(comparison[label]))
THINKING_LEVEL = "LOW"
experiment.update(tool_ablation={"creation_proposal":tool_proposal,
    "creation_usage":creation_usage, "tool_development":low_tool_dev,
    "efficiency_retained_on_development":efficiency_retained,
    "held_out":comparison, "summary":{k:costs(v) for k,v in comparison.items()}},
    calls=calls, usage=usage)
print("All calls:",len(calls),"| reported tokens:",sum(u['tokens'] or 0 for u in usage))


Generated tool: def split_bill(subtotal, tax, tip, diners, tip_on):
    sub = Decimal(subtotal)
    tax_rate = Decimal(tax) / Decimal('100')
    tip_rate = Decimal(tip) / Decimal('100')
    tax_amt = sub * tax_rate
    tip_base = sub + tax_amt if tip_on == 'total' else sub
    tip_amt = tip_base * tip_rate
    total = sub + tax_amt + tip_amt
    per_person = total / Decimal(diners)
    return per_person.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
Creation usage: {'tokens': 14002, 'prompt_tokens': 13702, 'output_tokens': 300, 'thinking_tokens': None, 'cached_tokens': None, 'seconds': 3.825}
LOW tool development: {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 43, 'api_errors': 0, 'tokens': 20734, 'median_seconds': 3.44, 'prompt_tokens': 7538, 'prompt_tokens_reported_attempts': 48, 'output_tokens': 4799, 'output_tokens_reported_attempts': 48, 'thinking_tokens': 8397, 'thinking_tokens_reported_attempts': 27, 'tokens_reported_attempts': 48, 'unattributed_tokens': 8397} | e

In [16]:
from IPython.display import display, JSON
display(JSON(experiment, expanded=False))


## 6. Let the improver see the additional failures

Both initial proposals chose no change. Their additional development attempts exposed two arithmetic failures under the same policy. This fixed improver now sees all 144 development attempts and may choose no change, instructions, or a tool. No held-out evidence enters the proposal. This is another task-agent improvement attempt, not recursive improvement. The development selection compares success rates with different sample counts; interpret it as preliminary.


In [19]:

# The first two proposals chose no_change; their development repetitions therefore
# provide additional failures from the SAME task policy. Never read held-out data.
baseline_dev = [r for name,rows in experiment['development'].items()
                if name=='original' or experiment['proposals'][name]['change']=='no_change'
                for r in rows]
print('Expanded original-policy development evidence:',summary(baseline_dev))
open_proposal = propose('no_change, instruction, or tool')
artifact = BASELINE if 'validation_error' in open_proposal else open_proposal
new_dev = evaluate(artifact, development)
base_rate = sum(r['correct'] for r in baseline_dev)/len(baseline_dev)
new_rate = sum(r['correct'] for r in new_dev)/len(new_dev)
retained = (not any('api_error' in r for r in baseline_dev+new_dev) and new_rate>base_rate)
print('Expanded-evidence candidate development:',summary(new_dev),'| retained:',retained)
new_test = evaluate(artifact, held_out)
print('Expanded-evidence candidate held-out:',summary(new_test))
experiment['expanded_evidence_improvement']={'development_evidence_attempts':len(baseline_dev),
    'proposal':{k:v for k,v in open_proposal.items() if k!='tool'},
    'development':new_dev,'retained_on_development':retained,'held_out':new_test}
experiment.update(calls=calls,usage=usage)
print('All calls:',len(calls),'| reported tokens:',sum(u['tokens'] or 0 for u in usage))


Expanded original-policy development evidence: {'correct': 142, 'attempts': 144, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 53889, 'median_seconds': 3.75}
tool | In fractional-06 repeat 1, the model failed due to mental arithmetic errors in summing and dividing fractional tax and tip (317.90 vs 317.86). A dedicated split_bill tool ensures exact Decimal precision and deterministic rounding.
Expanded-evidence candidate development: {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 47, 'api_errors': 0, 'tokens': 15799, 'median_seconds': 2.73} | retained: True
Expanded-evidence candidate held-out: {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 48, 'api_errors': 0, 'tokens': 16393, 'median_seconds': 2.76}
All calls: 580 | reported tokens: 357871


In [21]:
display(JSON(experiment, expanded=False))


Expand the complete experiment, including failures, proposals and token records.